# Narraciones con tu voz · Guía Clase N°1 · M1

Este cuaderno genera las narraciones de los 40 videos **con tu voz**, usando el modelo gratuito y de código abierto *Chatterbox Multilingual* (licencia MIT).

**Antes de empezar:** menú **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar**.

Luego: **Entorno de ejecución → Ejecutar todo**. Tu voz ya está cargada (paso 3). El cuaderno te dejará escuchar una prueba (paso 5) y al final descargará `audios.zip`.

## 1. Revisar que haya GPU

In [ ]:
import subprocess
r = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit("No hay GPU. Ve a Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4, y vuelve a ejecutar todo.")
print("GPU lista:", r.stdout.strip())

## 2. Instalar el modelo (tarda 2–4 minutos)
La primera vez, al terminar la instalación **la sesión se reinicia sola** (es normal). Entonces vuelve a usar **Entorno de ejecución → Ejecutar todo**: la segunda vez la instalación se salta y sigue directo.

In [ ]:
import importlib.metadata as md, os, subprocess
def version(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
sh = lambda cmd: subprocess.run(cmd, shell=True).returncode
reiniciar = False
if version("chatterbox-tts") is None:
    sh("pip install -q chatterbox-tts")
    reiniciar = True
# chatterbox usa torch 2.6.0: torchvision debe ser la versión compatible (0.21.0)
tv = version("torchvision")
if tv is not None and not tv.startswith("0.21"):
    if sh("pip install -q torchvision==0.21.0") != 0:
        sh("pip uninstall -y -q torchvision")
    reiniciar = True
if reiniciar:
    print("Instalación lista. La sesión se reiniciará sola para aplicar los cambios.")
    print("Cuando aparezca el aviso, vuelve a: Entorno de ejecución → Ejecutar todo.")
    os.kill(os.getpid(), 9)
else:
    print("Todo instalado y compatible")

## 3. Tu voz de referencia
Ya está cargada automáticamente desde tu repositorio (tomada de tu video). **No necesitas subir nada.** Si algún día quieres usar otra grabación, cambia `SUBIR_OTRA = False` por `True`.

In [ ]:
import os, subprocess, urllib.request
SUBIR_OTRA = False
if SUBIR_OTRA:
    from google.colab import files
    subido = files.upload(); nombre = list(subido.keys())[0]
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",nombre,"-af","silenceremove=start_periods=1:start_threshold=-45dB",
                    "-ac","1","-ar","24000","-t","25","voz_ref.wav"], check=True)
elif not os.path.exists("voz_ref.wav"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/voz_ref.wav", "voz_ref.wav")
print("Audio de referencia listo")
from IPython.display import Audio, display
display(Audio("voz_ref.wav"))

## 4. Cargar el modelo de español latinoamericano y los guiones
Se usa la versión del modelo entrenada para **español de Latinoamérica**, que evita el acento rioplatense. Si quisieras comparar con el modelo general, cambia `MODELO = "latam"` por `MODELO = "general"`.

In [ ]:
import json, urllib.request, shutil, re, torch, torchaudio as ta
from pathlib import Path
from huggingface_hub import snapshot_download
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

MODELO = "latam"        # "latam" = español latinoamericano · "general" = modelo multilingüe
EXAG, TEMP = 0.45, 0.75 # entonación más sobria y estable

ck = Path("/content/modelo_" + MODELO)
if not (ck / "listo.txt").exists():
    ck.mkdir(exist_ok=True)
    base_files = ["ve.pt", "s3gen.pt", "grapheme_mtl_merged_expanded_v1.json", "conds.pt", "Cangjie5_TC.json"]
    if MODELO != "latam":
        base_files.append("t3_mtl23ls_v2.safetensors")
    base = Path(snapshot_download("ResembleAI/chatterbox", allow_patterns=base_files))
    for f in base_files:
        if (base / f).exists():
            shutil.copy(base / f, ck / f)
    if MODELO == "latam":
        pack = Path(snapshot_download("ResembleAI/Chatterbox-Multilingual-es-mx-latam",
                    allow_patterns=["t3_es_mx_latam.safetensors", "grapheme_mtl_merged_expanded_v1.json", "s3gen_v3.pt"]))
        shutil.copy(pack / "t3_es_mx_latam.safetensors", ck / "t3_mtl23ls_v2.safetensors")
        shutil.copy(pack / "grapheme_mtl_merged_expanded_v1.json", ck / "grapheme_mtl_merged_expanded_v1.json")
        shutil.copy(ck / "s3gen.pt", ck / "s3gen_base.pt")
        if (pack / "s3gen_v3.pt").exists():
            shutil.copy(pack / "s3gen_v3.pt", ck / "s3gen.pt")
    (ck / "listo.txt").write_text("ok")

try:
    model = ChatterboxMultilingualTTS.from_local(ck, "cuda")
except Exception as err:
    print("Aviso: se usará el decodificador base (", type(err).__name__, ")")
    if (ck / "s3gen_base.pt").exists():
        shutil.copy(ck / "s3gen_base.pt", ck / "s3gen.pt")
    model = ChatterboxMultilingualTTS.from_local(ck, "cuda")

GUIONES = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/profcristobalespinoza-netizen/m1-clase-1/main/voz/guiones.json"))
print("Modelo", MODELO, "cargado ·", len(GUIONES), "ejercicios")

def frases(texto):
    partes = re.split(r'(?<=[.!?:])\s+', texto.strip())
    out, buf = [], ""
    for p in partes:
        if len(buf) + len(p) < 220: buf = (buf + " " + p).strip()
        else:
            if buf: out.append(buf)
            buf = p
    if buf: out.append(buf)
    return out

def generar_frase(f):
    """Genera una frase y revisa su duración; si sale cortada o con ruido extra, la rehace (hasta 4 intentos)."""
    esperado = len(f) / 15.5
    mejor = None
    for intento in range(4):
        wav = model.generate(f, language_id="es", audio_prompt_path="voz_ref.wav", exaggeration=EXAG, temperature=TEMP).cpu()
        d = wav.shape[-1] / model.sr
        if 0.55 * esperado <= d <= 1.9 * esperado + 0.8:
            return wav
        if mejor is None or abs(d - esperado) < mejor[0]:
            mejor = (abs(d - esperado), wav)
    return mejor[1]

def narrar(texto, destino):
    pausa = torch.zeros(1, int(model.sr * 0.30))
    trozos = []
    for f in frases(texto):
        trozos += [generar_frase(f), pausa]
    ta.save(destino, torch.cat(trozos, dim=1), model.sr)

## 5. Prueba: escucha cómo suena
Si no te convence, sube otro audio de referencia (borra `voz_ref.wav` en la carpeta de la izquierda y vuelve a ejecutar el paso 3).

In [ ]:
narrar(GUIONES["1"][0], "prueba.wav")
display(Audio("prueba.wav"))

## 6. Generar los 40 ejercicios (30–60 minutos)
No cierres la pestaña. Si se corta la conexión, vuelve a ejecutar esta celda: continúa donde quedó (mientras la sesión siga activa).

In [ ]:
import os, time
os.makedirs("audios_v2", exist_ok=True)
total = sum(len(v) for v in GUIONES.values()); hechos = 0; t0 = time.time()
for n in sorted(GUIONES, key=int):
    for k, texto in enumerate(GUIONES[n]):
        destino = f"audios_v2/ej{int(n):02d}_{k:02d}.wav"
        if not os.path.exists(destino):
            narrar(texto, destino)
        hechos += 1
    print(f"Ejercicio {n} listo · {hechos}/{total} tramos · {int(time.time()-t0)//60} min")
print("Todas las narraciones generadas")

## 7. Descargar
Se descarga `audios.zip`. Súbelo al chat con Claude para armar los videos.

In [ ]:
import shutil
shutil.make_archive("audios", "zip", "audios_v2")
from google.colab import files
files.download("audios.zip")